# Eval v2: Turn Detector with Threshold Classification
- P(im_end) > 0.5 → positive (turn complete)
- P(im_end) < 0.5 → negative (turn incomplete)

In [10]:
import torch
import math
import json
import numpy as np
import torch.nn.functional as F
from chinidataset import StreamingDataset
from transformers import AutoTokenizer, Qwen3ForCausalLM
from liger_kernel.transformers import apply_liger_kernel_to_qwen3, LigerFusedLinearCrossEntropyLoss
from tqdm import tqdm
import glob
import os
import pandas as pd

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Scicom-intl/turn-detector-Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16).cuda().eval()

## Config

In [12]:
TEST_FILE = './parquet-test-synthetic'
THRESHOLD = 0.5

## Load Model

In [13]:
apply_liger_kernel_to_qwen3(
    rope=True,
    swiglu=True,
    rms_norm=True,
    cross_entropy=False,
    fused_linear_cross_entropy=False,
)

class Model(Qwen3ForCausalLM):
    def __init__(self, config):
        super().__init__(config)
        self.loss = LigerFusedLinearCrossEntropyLoss(reduction='sum')

    def forward(self, input_ids, attention_mask=None, position_ids=None,
                labels=None, **kwargs):
        super_out = self.model.forward(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            **kwargs,
        )
        if labels is not None:
            embeddings = super_out.last_hidden_state
            embeddings = embeddings[:, :-1].reshape(-1, embeddings.shape[-1])
            labels = labels[..., 1:].contiguous().reshape(-1)
            loss = self.loss(self.lm_head.weight, embeddings, labels)
            loss = loss / (labels != -100).sum()
            return {'loss': loss, 'hidden_states': super_out.last_hidden_state}
        return super_out

model = Model.from_pretrained(
    model_id,
    attn_implementation='flash_attention_2',
    torch_dtype='bfloat16',
)
model.eval()
model.cuda()

Model(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): LigerRMSNorm((128,), eps=1e-06, offset=0.0, in_place=True, row_mode=None)
          (k_norm): LigerRMSNorm((128,), eps=1e-06, offset=0.0, in_place=True, row_mode=None)
        )
        (mlp): LigerSwiGLUMLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        )
        (input_layernorm): 

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
IM_END_TOKEN_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
print(f'<|im_end|> token ID: {IM_END_TOKEN_ID}')

<|im_end|> token ID: 151645


## P(im_end) function

In [16]:
def get_im_end_prob(text):
    """Get probability that the turn should end.
    Strips trailing <|im_end|> so the model predicts whether to emit it.
    """
    ix = text.rfind('<|im_end|>')
    if ix != -1 and ix == len(text) - len('<|im_end|>'):
        text = text[:ix]
    
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = Qwen3ForCausalLM.forward(model, **inputs).logits
    logprob = F.log_softmax(logits[0, -1], dim=-1)[IM_END_TOKEN_ID].item()
    prob = math.exp(logprob)
    return prob, logprob

## Run on test set

In [17]:
ds = StreamingDataset(local=TEST_FILE)
print(f'Test samples: {len(ds)}')

results = []
for i in tqdm(range(len(ds))):
    row = ds[i]
    meta = json.loads(row['text'])
    true_label = meta['label']
    language = meta.get('language', 'unknown')
    
    # Decode tokens to text
    text = tokenizer.decode(row['input_ids'].astype(np.int64), skip_special_tokens=False)
    
    prob, logprob = get_im_end_prob(text)
    pred_label = 'positive' if prob > THRESHOLD else 'negative'
    correct = pred_label == true_label
    
    results.append({
        'idx': i,
        'true_label': true_label,
        'pred_label': pred_label,
        'correct': correct,
        'prob': prob,
        'logprob': logprob,
        'language': language,
        'text_preview': text[:200],
    })

df = pd.DataFrame(results)
print(f'\nDone: {len(df)} samples evaluated')

Test samples: 238


100%|██████████| 238/238 [00:06<00:00, 37.77it/s]


Done: 238 samples evaluated


## Overall accuracy

In [18]:
accuracy = df['correct'].mean()
print(f'Threshold: {THRESHOLD}')
print(f'Overall accuracy: {accuracy:.4f} ({df["correct"].sum()}/{len(df)})')

# Per class
for label in ['positive', 'negative']:
    subset = df[df['true_label'] == label]
    acc = subset['correct'].mean()
    print(f'  {label}: {acc:.4f} ({subset["correct"].sum()}/{len(subset)})')

Threshold: 0.5
Overall accuracy: 0.8529 (203/238)
  positive: 0.7059 (84/119)
  negative: 1.0000 (119/119)


## Per language accuracy

In [19]:
print(f'Threshold: {THRESHOLD}\n')
for lang in sorted(df['language'].unique()):
    subset = df[df['language'] == lang]
    acc = subset['correct'].mean()
    pos = subset[subset['true_label'] == 'positive']
    neg = subset[subset['true_label'] == 'negative']
    pos_acc = pos['correct'].mean() if len(pos) > 0 else 0
    neg_acc = neg['correct'].mean() if len(neg) > 0 else 0
    print(f'{lang}:')
    print(f'  overall:  {acc:.4f} ({subset["correct"].sum()}/{len(subset)})')
    print(f'  positive: {pos_acc:.4f} ({pos["correct"].sum()}/{len(pos)})')
    print(f'  negative: {neg_acc:.4f} ({neg["correct"].sum()}/{len(neg)})')

Threshold: 0.5

chinese-english:
  overall:  0.8500 (17/20)
  positive: 0.7000 (7/10)
  negative: 1.0000 (10/10)
chinese-malay:
  overall:  0.8500 (17/20)
  positive: 0.7000 (7/10)
  negative: 1.0000 (10/10)
chinese-tamil:
  overall:  0.9000 (18/20)
  positive: 0.8000 (8/10)
  negative: 1.0000 (10/10)
english-chinese:
  overall:  0.8000 (16/20)
  positive: 0.6000 (6/10)
  negative: 1.0000 (10/10)
english-malay:
  overall:  0.9000 (18/20)
  positive: 0.8000 (8/10)
  negative: 1.0000 (10/10)
english-tamil:
  overall:  0.9000 (18/20)
  positive: 0.8000 (8/10)
  negative: 1.0000 (10/10)
malay-chinese:
  overall:  1.0000 (20/20)
  positive: 1.0000 (10/10)
  negative: 1.0000 (10/10)
malay-english:
  overall:  1.0000 (20/20)
  positive: 1.0000 (10/10)
  negative: 1.0000 (10/10)
malay-tamil:
  overall:  0.8500 (17/20)
  positive: 0.7000 (7/10)
  negative: 1.0000 (10/10)
tamil-chinese:
  overall:  0.9444 (17/18)
  positive: 0.8889 (8/9)
  negative: 1.0000 (9/9)
tamil-english:
  overall:  0.6500

## Confusion matrix

In [20]:
tp = len(df[(df['true_label'] == 'positive') & (df['pred_label'] == 'positive')])
tn = len(df[(df['true_label'] == 'negative') & (df['pred_label'] == 'negative')])
fp = len(df[(df['true_label'] == 'negative') & (df['pred_label'] == 'positive')])
fn = len(df[(df['true_label'] == 'positive') & (df['pred_label'] == 'negative')])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'Threshold: {THRESHOLD}')
print(f'\nConfusion Matrix:')
print(f'                 Predicted')
print(f'                 Pos    Neg')
print(f'  Actual Pos     {tp:<6} {fn}')
print(f'  Actual Neg     {fp:<6} {tn}')
print(f'\nPrecision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

Threshold: 0.5

Confusion Matrix:
                 Predicted
                 Pos    Neg
  Actual Pos     84     35
  Actual Neg     0      119

Precision: 1.0000
Recall:    0.7059
F1:        0.8276


## Probability distribution

In [21]:
print('Positive samples (should be > 0.5):')
pos_probs = df[df['true_label'] == 'positive']['prob']
print(f'  mean={pos_probs.mean():.4f}  median={pos_probs.median():.4f}  min={pos_probs.min():.4f}  max={pos_probs.max():.4f}')

print('\nNegative samples (should be < 0.5):')
neg_probs = df[df['true_label'] == 'negative']['prob']
print(f'  mean={neg_probs.mean():.4f}  median={neg_probs.median():.4f}  min={neg_probs.min():.4f}  max={neg_probs.max():.4f}')

Positive samples (should be > 0.5):
  mean=0.6775  median=0.7973  min=0.0001  max=0.9973

Negative samples (should be < 0.5):
  mean=0.0000  median=0.0000  min=0.0000  max=0.0001


## Threshold sweep

In [22]:
print(f'{"Threshold":<12} {"Accuracy":<10} {"Precision":<10} {"Recall":<10} {"F1":<10}')
print('-' * 52)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    df['_pred'] = df['prob'].apply(lambda x: 'positive' if x > t else 'negative')
    acc = (df['_pred'] == df['true_label']).mean()
    _tp = len(df[(df['true_label'] == 'positive') & (df['_pred'] == 'positive')])
    _fp = len(df[(df['true_label'] == 'negative') & (df['_pred'] == 'positive')])
    _fn = len(df[(df['true_label'] == 'positive') & (df['_pred'] == 'negative')])
    _prec = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0
    _rec = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0
    _f1 = 2 * _prec * _rec / (_prec + _rec) if (_prec + _rec) > 0 else 0
    marker = ' <<<' if t == THRESHOLD else ''
    print(f'{t:<12} {acc:<10.4f} {_prec:<10.4f} {_rec:<10.4f} {_f1:<10.4f}{marker}')
df.drop('_pred', axis=1, inplace=True)

Threshold    Accuracy   Precision  Recall     F1        
----------------------------------------------------
0.1          0.9580     1.0000     0.9160     0.9561    
0.2          0.9286     1.0000     0.8571     0.9231    
0.3          0.9160     1.0000     0.8319     0.9083    
0.4          0.8782     1.0000     0.7563     0.8612    
0.5          0.8529     1.0000     0.7059     0.8276     <<<
0.6          0.8277     1.0000     0.6555     0.7919    
0.7          0.7983     1.0000     0.5966     0.7474    
0.8          0.7479     1.0000     0.4958     0.6629    
0.9          0.6849     1.0000     0.3697     0.5399    


## Worst predictions

In [23]:
wrong = df[~df['correct']].sort_values('prob', ascending=False)
print(f'Wrong predictions: {len(wrong)}/{len(df)}')
print()
for _, row in wrong.head(10).iterrows():
    print(f'[{row["true_label"]}→{row["pred_label"]}] prob={row["prob"]:.4f} lang={row["language"]}')
    print(f'  {row["text_preview"][:150]}')
    print()

Wrong predictions: 35/238

[positive→negative] prob=0.4989 lang=malay-tamil
  <|im_start|>assistance
Halo, saya ada masalah dengan pembayaran saya. Saya dah cuba bayar tetapi tak berjaya. Kenapa ini terjadi? Aiyoh, sangat frustr

[positive→negative] prob=0.4817 lang=malay-tamil
  <|im_start|>assistance
Halo, saya nak minta refund untuk barang yang saya beli minggu lepas. Tak berfungsi langsung, sangat frust lah. Can you help me

[positive→negative] prob=0.4817 lang=chinese-english
  <|im_start|>assistance
你好，我想开一个新账户，能帮我吗？我不太确定这个过程是怎样的。<|im_end|>
<|im_start|>assistant
你好！当然可以帮您。开新账户的流程很简单。您需要提供一些基本资料，比如您的身份证和联系方式。请问您是要开个人账户还是商业账户？

[positive→negative] prob=0.4352 lang=chinese-malay
  <|im_start|>assistance
你好，我想要开一个新账户，可以帮我吗？我不太确定需要什么文件。<|im_end|>
<|im_start|>assistant
你好！当然可以帮助你。开新账户通常需要身份证和住址证明。你有没有这些文件呢？

[positive→negative] prob=0.4267 lang=english-chinese
  <|im_start|>assistance
Hello, 我想问一下关于我的保险索赔的情况。上个月我提交了申请，但还没有收到回复。<|im_end|>
<|im_start|>assistant
Hi! I understand your conc

## Manual test

In [24]:
test_prompts = [
    # Complete - should be > 0.5
    '<|im_start|>user\nHello, saya nak tanya pasal bil saya.<|im_end|>\n<|im_start|>assistant\nBoleh, sila berikan nombor akaun anda.',
    # Incomplete - should be < 0.5
    '<|im_start|>user\nHello, saya nak tanya pasal bil saya.<|im_end|>\n<|im_start|>assistant\nBoleh, sila berikan nombor',
    # Complete
    '<|im_start|>user\nTerima kasih banyak atas bantuan anda.',
    # Incomplete
    '<|im_start|>user\nSaya nak tanya pasal',
]

for prompt in test_prompts:
    prob, lp = get_im_end_prob(prompt)
    label = 'positive' if prob > THRESHOLD else 'negative'
    print(f'[{label:>8}] prob={prob:.4f}')

[positive] prob=0.7359
[negative] prob=0.0000
[negative] prob=0.0000
[negative] prob=0.0000
